# Capstone --- Chapter 14: Human-in-the-Loop and Escalation

Chapter~14 concerns the point at which an autonomous agent stops acting and hands off to a human: the conditions that warrant escalation, and the record the agent produces so that a reviewer can decide without re-running the case. This companion reads that principle on the capstone banking complaint agent, which escalates along two distinct routes and carries, in each, enough context for a human decision.

The capstone escalates on two kinds of condition. The first is a *gate refusal*: a governance gate on the executor rejects a proposed tool call --- personally identifiable information detected in an argument, or a prompt-injection pattern in the message. The second is *regulatory risk*: the `flag_regulatory` tool marks a complaint as implicating UDAAP or Regulation~X, and the agent hands the case to a human rather than draft an automated reply. The two routes reach a human through different mechanisms, examined in turn below.

In [ ]:
from agentlab.governance.escalation import (
    EscalationRequest,
    HumanResponse,
    HumanDecision,
    ScriptedReviewer,
    HumanReviewer,
)
from agentlab.governance.policies import pii_policy, prompt_injection_policy
from agentlab.tools.executor import GateDecision
from agentlab.core.action import ToolCall, Escalate

## The escalation record

An escalation is not a bare signal; it is a record. `EscalationRequest` carries the `run_id` and `step` that locate the case in the audit log, a `reason` in plain language, the `proposed_action` that triggered the handoff, and the `gate_results` that decided it. The record serializes to JSON, so it survives being written to a queue and read back by a reviewer who was not present when the agent ran.

In [ ]:
request = EscalationRequest(
    run_id='case-4417',
    step=0,
    reason='pii_policy: detected PII: ssn',
    proposed_action={'kind': 'tool_call', 'tool_name': 'classify_complaint',
                     'arguments': {'message': 'my SSN is 123-45-6789 and I was overcharged'}},
    gate_results=[{'gate': 'pii_policy', 'decision': 'escalate',
                   'reason': 'detected PII: ssn'}],
)
print(request.to_json(indent=2))

The record round-trips: `from_json` reconstructs an equal request from the serialized form, which is the property a durable review queue requires.

In [ ]:
restored = EscalationRequest.from_json(request.to_json())
print('round-trips equal:', restored == request)

## Route one: a gate refusal escalates

The governance gates on the executor decide, per proposed tool call, whether the call may proceed. A gate returns a `GateResult` carrying a `GateDecision`: `ALLOW` lets the call run, `DENY` blocks it, and `ESCALATE` requests a human. The PII gate escalates when an argument contains a social-security or card number; the prompt-injection gate denies a call whose arguments carry an override instruction. Both are read below on a message that carries a social-security number.

In [ ]:
call = ToolCall(
    tool_name='classify_complaint',
    arguments={'message': 'my SSN is 123-45-6789 and I was overcharged a $35 fee'},
)
pii = pii_policy(call, None)
inj = prompt_injection_policy(call, None)
print(f'{pii.gate_name:24s} {pii.decision.value:9s} {pii.reason}')
print(f'{inj.gate_name:24s} {inj.decision.value:9s} {inj.reason or "(clean)"}')

A gate that returns `ESCALATE` does not itself contact a human. The governance harness translates the escalating gate result into an `EscalationRequest` and routes it to a `HumanReviewer` --- the boundary between the automated system and the person. The harness builds exactly the record shown below from the gate's decision and the proposed action.

In [ ]:
escalating = [g for g in (pii, inj) if g.decision == GateDecision.ESCALATE]
reason = '; '.join(f'{g.gate_name}: {g.reason}' for g in escalating)
gate_refusal = EscalationRequest(
    run_id='case-4417',
    step=0,
    reason=reason,
    proposed_action=call.model_dump(),
    gate_results=[{'gate': g.gate_name, 'decision': g.decision.value, 'reason': g.reason}
                  for g in (pii, inj)],
)
print(gate_refusal.to_json(indent=2))

## The reviewer returns a decision

A `HumanReviewer` maps an `EscalationRequest` to a `HumanResponse`, whose `decision` is one of `APPROVE`, `DENY` or `DEFER`. `ScriptedReviewer` supplies a fixed response and captures every request it sees, which is what makes an escalation path testable without a person in the loop; `CLIReviewer` reads the decision from standard input for interactive use. The scripted reviewer below denies the PII case, and the captured request confirms the reviewer saw the full record.

In [ ]:
reviewer = ScriptedReviewer(HumanResponse(HumanDecision.DENY, note='redact SSN before reprocessing'))
print('is a HumanReviewer:', isinstance(reviewer, HumanReviewer))
response = reviewer.review(gate_refusal)
print('decision :', response.decision.value)
print('note     :', response.note)
print('captured :', len(reviewer.requests), 'request(s);',
      'reason =', reviewer.requests[0].reason)

The harness acts on the decision. `APPROVE` overrides the gate and forces the call; `DEFER` ends the run in escalated status, leaving the case for a human; `DENY` keeps the refusal in place. The decision is recorded in the audit log alongside the request, so the disposition of every escalated case is reconstructable after the fact.

## Route two: regulatory risk escalates from within the workflow

The second route does not pass through the executor's gates. When `flag_regulatory` marks a complaint as implicating UDAAP or Regulation~X, the complaint agent emits an `Escalate` action directly, in place of proposing the `draft_response` call. A regulated matter is handed to a human rather than answered by the model. The `Escalate` action carries the same two elements as the gate route --- a plain-language `reason` and a structured `context` --- so the two routes converge on one auditable handoff shape.

In [ ]:
# What the agent constructs when flag_regulatory reports escalate=True.
flags = {'flags': ['UDAAP'], 'escalate': True}
regulatory = Escalate(
    reason=f"regulatory risk flagged: {flags['flags']}",
    context={'flags': flags['flags']},
)
print('kind    :', regulatory.kind)
print('reason  :', regulatory.reason)
print('context :', regulatory.context)

## The two routes side by side

The gate route escalates *before* a tool acts, on a property of the proposed call (PII, injection); the regulatory route escalates *within* the workflow, on the result of a tool (`flag_regulatory`). Both stop autonomous action and produce a record a human can act on. The table below summarizes the routing the capstone implements.

In [ ]:
routes = [
    ('PII in argument',        'pii_policy',              'ESCALATE (gate refusal)', 'EscalationRequest -> HumanReviewer'),
    ('prompt-injection intent','prompt_injection_policy', 'DENY (gate refusal)',     'call blocked; no autonomous action'),
    ('UDAAP / Reg_X flagged',  'flag_regulatory',         'Escalate action',          'agent hands off in place of draft'),
]
print(f'{"condition":26s} {"decided by":24s} {"outcome":26s} handoff')
for cond, by, outcome, handoff in routes:
    print(f'{cond:26s} {by:24s} {outcome:26s} {handoff}')

This is the capstone's realization of Chapter~14: escalation is a defined boundary, not an exception. A gate refusal or a regulatory flag stops the agent, and the handoff is a structured record --- an `EscalationRequest` routed to a `HumanReviewer`, or an `Escalate` action carrying its reason and context --- that a person can decide on and an auditor can reconstruct. Chapter~16 assembles these routes with the five tools into the governed workflow the capstone runs end to end.